# Deep Learning — Fondations réseaux & PyTorch (**Exercices guidés + indices**)

Ce notebook contient **les cellules à compléter** + **des indices** juste au-dessus de chaque question.

Thèmes :
1) Initialisation des poids  
2) BatchNorm / LayerNorm  
3) Dropout  
4) Gradient vanishing / exploding  
5) Dataset & DataLoader  
6) GPU / CUDA  
7) Sauvegarde / chargement modèle  
8) Training vs Inference mode  


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

print('Torch:', torch.__version__)


Torch: 2.10.0+cpu


## 1 — Initialisation des poids

**But :** initialiser une couche pour stabiliser l’entraînement.

### Indices
- Une couche linéaire : `nn.Linear(in_features, out_features)`
- Les fonctions d'init sont dans `torch.nn.init` (alias : `nn.init`)
- Xavier uniforme : `nn.init.xavier_uniform_(layer.weight)`
- Pour vérifier : variance `layer.weight.var()`


In [4]:
layer = nn.Linear(100,100)

# TODO: initialiser les poids avec Xavier uniform
nn.init.xavier_uniform_(layer.weight)
# TODO: afficher la variance des poids
layer.weight.var()

tensor(0.0100, grad_fn=<VarBackward0>)

## 2 — BatchNorm vs LayerNorm

**But :** comparer la normalisation BatchNorm et LayerNorm.

### Indices
- BatchNorm1d : `nn.BatchNorm1d(10)`

- LayerNorm : `nn.LayerNorm(10)`

- Appliquer : `y = layer(x)`

- Mesurer : `y.mean()` et `y.std()`

- Attendu : les sorties sont centrées/réduites (approx).


In [7]:
x = torch.randn(32, 10)

# ✅ À COMPLÉTER
# TODO: créer BatchNorm1d(10)
bn = nn.BatchNorm1d(10)

# TODO: créer LayerNorm(10)
ln = nn.LayerNorm(10)

# TODO: appliquer bn et ln à x
y_bn = bn(x)
y_ln = ln(x)

# TODO: afficher mean et std pour les deux
y_bn.mean(), y_bn.std(),y_ln.mean(),y_ln.std()


(tensor(-1.0431e-08, grad_fn=<MeanBackward0>),
 tensor(1.0016, grad_fn=<StdBackward0>),
 tensor(-4.4703e-09, grad_fn=<MeanBackward0>),
 tensor(1.0016, grad_fn=<StdBackward0>))

## 3 — Dropout

**But :** voir que Dropout agit seulement en mode entraînement.

### Indices
- Mode entraînement : `drop.train()`

- Mode évaluation : `drop.eval()`

- En train : certaines valeurs deviennent 0

- En eval : la sortie reste identique (pas de masquage)


In [8]:
drop = nn.Dropout(p=0.5)
x = torch.ones(10)

# ✅ À COMPLÉTER
# TODO: passer en mode train
drop.train()
# TODO: appliquer dropout et afficher
y_train = drop(x)
y_train
# TODO: passer en mode eval
drop.eval()
# TODO: appliquer dropout et afficher
y_eval=drop(x)
y_eval



tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

## 4 — Observer les gradients (vanishing / exploding)

**But :** calculer une loss et observer la norme des gradients des paramètres.

### Indices
- Forward : `y_hat = model(x)`

- Loss : `F.mse_loss(y_hat, y)`

- Backward : `loss.backward()`

- Norme gradient : `p.grad.norm()`

- Astuce : imprimez `name` + la norme.


In [10]:
model = nn.Sequential(
    nn.Linear(50, 100), nn.ReLU(),
    nn.Linear(100, 100), nn.ReLU(),
    nn.Linear(100, 1)
)

x = torch.randn(32, 50)
y = torch.randn(32, 1)

# ✅ À COMPLÉTER
# TODO: forward
y_hat=model(x)
# TODO: loss MSE
loss=F.mse_loss(y_hat,y)
# TODO: backward
loss.backward()

# TODO: afficher norme des gradients pour chaque paramètre
for name, p in model.named_parameters():
  print(name,p.grad.norm())

0.weight tensor(0.5024)
0.bias tensor(0.0762)
2.weight tensor(0.6411)
2.bias tensor(0.1253)
4.weight tensor(0.5346)
4.bias tensor(0.1590)


## 5 — Dataset & DataLoader

**But :** créer un dataset et itérer en mini-batch.

### Indices
- Dataset : `TensorDataset(x, y)`

- Loader : `DataLoader(dataset, batch_size=32, shuffle=True)`

- Itérer : `for xb, yb in loader:`

- Afficher `xb.shape` et `yb.shape`


In [13]:
x = torch.randn(1000, 10)
y = torch.sum(x, dim=1, keepdim=True)

# ✅ À COMPLÉTER
# TODO: créer TensorDataset
dataset = TensorDataset(x,y)

# TODO: créer DataLoader(batch_size=32, shuffle=True)
loader = DataLoader(dataset,batch_size=32,shuffle=True)

# TODO: afficher shape du premier batch
for xb,yb in loader:
  print(xb.shape,yb.shape)
  break

torch.Size([32, 10]) torch.Size([32, 1])


## 6 — GPU / CUDA

**But :** exécuter un forward sur GPU si disponible.

### Indices
- Tester : `torch.cuda.is_available()`

- Device : `torch.device('cuda')` ou `torch.device('cpu')`

- Envoyer : `model.to(device)` et `x.to(device)`

- Piège : modèle et données doivent être sur le **même** device.


In [14]:
# ✅ À COMPLÉTER
# TODO: détecter device cuda ou cpu
device = torch.cuda.is_available() and torch.device('cuda') or torch.device('cpu')

# TODO: créer Linear(10,1) et envoyer sur device
model = model.to(device)

# TODO: créer x sur device et faire un forward, puis afficher y.device
x.to(device)


tensor([[-0.9087, -1.5738, -1.0527,  ...,  0.2260, -0.8421,  0.4334],
        [ 1.6206,  0.1471, -2.2376,  ...,  1.3370, -0.8070,  0.1437],
        [-0.2329, -0.0813, -1.1694,  ...,  0.7635, -1.7328, -0.1218],
        ...,
        [ 0.3812,  0.7034,  0.2668,  ..., -1.1773,  0.6875,  0.8045],
        [-0.3079,  0.2606, -0.9157,  ...,  0.1189, -0.0898, -0.1241],
        [-0.6981, -1.3556,  0.1507,  ..., -0.3749, -0.0028, -1.2926]])

## 7 — Sauvegarde / Chargement

**But :** sauvegarder les poids d’un modèle et les recharger dans un nouveau modèle.

### Indices
- Sauvegarder : `torch.save(model.state_dict(), 'model.pth')`

- Recharger :

  1) recréer la **même architecture**

  2) `model2.load_state_dict(torch.load('model.pth'))`

- Après chargement : souvent `model2.eval()`


In [15]:
# ✅ À COMPLÉTER
# TODO: créer modèle Linear(10,1)
model = nn.Linear(10,1)

# TODO: sauvegarder state_dict dans 'model.pth'
torch.save(model.state_dict(),'model.pth')
# TODO: recréer modèle et charger state_dict
model2 = nn.Linear(10,1)
model2.load_state_dict(torch.load('model.pth'))
# TODO: vérifier qu'un forward fonctionne
model2.eval()



Linear(in_features=10, out_features=1, bias=True)

## 8 — Training vs Inference mode

**But :** constater que BatchNorm et Dropout changent de comportement entre `train()` et `eval()`.

### Indices
- En `train()` : dropout masque, BatchNorm utilise stats du batch

- En `eval()` : dropout OFF, BatchNorm utilise stats apprises

- Faites deux forwards et comparez (`torch.allclose` par ex.)


In [16]:
model = nn.Sequential(
    nn.Linear(10, 10),
    nn.BatchNorm1d(10),
    nn.Dropout(0.5),
    nn.Linear(10, 1)
)

x = torch.randn(4, 10)

# ✅ À COMPLÉTER
# TODO: mode train + forward
model.train()
y_train = model(x)

# TODO: mode eval + forward
model.eval()
y_eval=model(x)

# TODO: comparer les sorties (afficher les deux + un test allclose)
print(y_train)
print(y_eval)
torch.allclose(y_train,y_eval)



tensor([[-1.1354],
        [ 0.7264],
        [-1.1437],
        [-0.1437]], grad_fn=<AddmmBackward0>)
tensor([[-0.7409],
        [-0.2650],
        [-0.7625],
        [ 0.0030]], grad_fn=<AddmmBackward0>)


False